# Preliminaries

## Imports

In [ ]:
from pathlib import Path

import holoviews as hv
import panel as pn

from bridge.providers.vision import Coco2017Detection

hv.extension("bokeh")
pn.extension()

TMP_NOTEBOOK_ROOT = Path("/tmp/bridge-ds/tutorials")

## Load Dataset

In [ ]:
root_dir = TMP_NOTEBOOK_ROOT / "coco"

provider = Coco2017Detection(root_dir, split="val", img_source="stream")
ds = provider.build_dataset()
ds

# LoadMechanism

In this tutorial we will learn about the **LoadMechanism**, Bridge's way of loading raw data from different sources.

A quick reminder: to access the raw data within each element, we need to use the **SampleAPI** with `sample.data / element.data`. The column `data` in the **TableAPI** usually (but not always) contains a reference to the data rather than the data itself:


In [ ]:
ds.elements.head(2)

When we want to access data for a given element, we need to call the `element.data` property. In COCO, every sample has elements grouped by role: an `"image"` element and zero or more `"bbox"` elements. We can fetch the image element via `sample.one("image")` and the list of bbox elements via `sample.elements["bbox"]`.

In [ ]:
sample = ds.iget(0)
img_element = sample.one("image")
print("img_data:", img_element.data.shape, "\n")

bbox_elements = sample.elements["bbox"]
print(*[bb.data for bb in bbox_elements], sep="\n")

Every element holds a **LoadMechanism**, an object responsible for loading data from different sources. In this case, for images, `element.data` will perform an HTTP request and load the image in the response. For bboxes, which already exist in-memory (note that we can see them directly in the `data` column of the bbox rows), `element.data` will simply load the stored object.

The **LoadMechanism** is defined by two variables:

In [ ]:
print("Image element, loaded over HTTP:")
print("url_or_data:", img_element._load_mechanism.url_or_data)
print("encoding:", img_element._load_mechanism.encoding)
print()
print("Bbox elements, loaded from memory:")
print("url_or_data:", bbox_elements[0]._load_mechanism.url_or_data)
print("encoding:", bbox_elements[0]._load_mechanism.encoding)

- **url_or_data**, as its name suggests, contains either a url that references the object (url broadly speaking - including local paths, s3 paths, etc.), or contains the actual object, in case we want to store it directly in-memory.
- **encoding** - accepts a string that is used to determine which logic is used to load the object. Should we load the image using PIL? or a text file using simple `with open()`? this value determines that. To find which encodings are supported, use `list_registered_encodings`.

## In summary
1. Bridge loads data lazily, only when `element.data` is called
2. The loading mechanism function accepts **url_or_data** which defines where to load from (or what to load), and **encoding** which defines _how_ to load it.